In [ ]:
%load_ext autoreload
%autoreload 2

### Updating PepCL with immunogenicity data to prioritize T cell epitopes

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT.name not in ("CLSBN26", "PepCL_Manuscript"):    
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise RuntimeError("Could not find CLSBN26 project root.")
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / "Data"
SRC_DIR = PROJECT_ROOT / "src"
MODEL_CHECKPOINTS_DIR = PROJECT_ROOT / "Model_Checkpoints"
FIGURES_DIR = PROJECT_ROOT / "Figures"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_DIR:", DATA_DIR)
print("FIGURES_DIR:", FIGURES_DIR)
print("PROJECT_ROOT in sys.path:", str(PROJECT_ROOT) in sys.path)

PROJECT_ROOT: /home/pchati/PepCL_Manuscript
DATA_DIR: /home/pchati/PepCL_Manuscript/Data
FIGURES_DIR: /home/pchati/PepCL_Manuscript/Figures
PROJECT_ROOT in sys.path: True


In [ ]:
# imports
from src.general_utils import *
from src.get_feature_data import *
from src.Model.mhcprime import *
from src.model_utils import *
from src.general_utils import *
from src.get_feature_data import *
from src.pepcl import *
from src.pepcl_multi import *
from src.mhcprime_inference import *
from src.plotting import *

# misc
import warnings
warnings.filterwarnings("ignore")
device="cuda"

In [ ]:
# change gpu
# GPU_ID = 0
# GPU_ID = 1
# GPU_ID = 2
GPU_ID = 3 # set above

torch.cuda.set_device(GPU_ID)
print(f"Using GPU: {torch.cuda.get_device_name(GPU_ID)} (ID: {torch.cuda.current_device()})")

Using GPU: NVIDIA H100 80GB HBM3 (ID: 3)


### Create figure directory

In [ ]:
# create figures and data dir
immunogenicity_figures_dir=f"{FIGURES_DIR}/Revisions/Immunogenicity_Analysis"
immunogenicity_figures_data_dir=f"{immunogenicity_figures_dir}/Associated_Data"

os.makedirs(immunogenicity_figures_dir, exist_ok=True)
os.makedirs(immunogenicity_figures_data_dir, exist_ok=True)

### Get datasets

#### Format pseudosequences and get feature data

In [ ]:
# format pseudosequences
from Data.mhc_pseudo_seqs import mhc_seq_dict
mhc_seq_dict={k.removeprefix("HLA-").replace("*","").replace(":",""):v for k,v in mhc_seq_dict.items()}

In [ ]:
# get feature table and processed features
aa_features_table_ext, descriptor_groups=load_feature_data()
processed_feature_table, corr_info = process_aa_feature_table(aa_features_table_ext)
feature_names = list(processed_feature_table.columns)  # pass to FeatureProjectionEncoder

#### MS train/val/test data

In [ ]:
# ms train and test sets
datasets_dir = DATA_DIR / "Training_Datasets" / "Processed" / "MS_Train_Test_Sets"
ms_train_data=pd.read_csv(f"{datasets_dir}/ms_train_data_remaining_minimal.csv", index_col=0)
ms_val_data=pd.read_csv(f"{datasets_dir}/ms_val_data.csv", index_col=0)
ms_test_data=pd.read_csv(f"{datasets_dir}/ms_test_data.csv", index_col=0)

In [ ]:
# process
ms_train_data=full_process_data(ms_train_data, mhc_seq_dict, remove_flank=True, add_pseudo_label=False)
ms_train_data=create_seq_allele_column(ms_train_data)

ms_val_data=full_process_data(ms_val_data, mhc_seq_dict, remove_flank=True, add_pseudo_label=False)
ms_val_data=create_seq_allele_column(ms_val_data)

ms_test_data=full_process_data(ms_test_data, mhc_seq_dict, remove_flank=True, add_pseudo_label=False)
ms_test_data=create_seq_allele_column(ms_test_data)

#### YD train/val/test data

 - Use primary validation split.

In [ ]:
# yd train and test sets
holec_datasets_dir = DATA_DIR / "Training_Datasets" / "Processed" / "Holec_et_al_2025"
holec_train_set = pd.read_csv(f"{holec_datasets_dir}/holec_train_set_remaining.csv", index_col=0)
holec_val_set = pd.read_csv(f"{holec_datasets_dir}/holec_val_set.csv", index_col=0)
holec_test_set  = pd.read_csv(f"{holec_datasets_dir}/holec_test_set.csv", index_col=0)

In [ ]:
# process
holec_train_set=full_process_data(holec_train_set, mhc_seq_dict, remove_flank=True, add_pseudo_label=False)
holec_train_set=create_seq_allele_column(holec_train_set)

holec_val_set=full_process_data(holec_val_set, mhc_seq_dict, remove_flank=True, add_pseudo_label=False)
holec_val_set=create_seq_allele_column(holec_val_set)

holec_test_set=full_process_data(holec_test_set, mhc_seq_dict, remove_flank=True, add_pseudo_label=False)
holec_test_set=create_seq_allele_column(holec_test_set)

In [ ]:
# get background as well
chp_background_10k_per_allele_x_len_processed_yd_opt = pd.read_pickle(f"{DATA_DIR}/chp_background_10k_per_allele_x_len_processed_yd_opt.pkl")

#### EpiScan R9 and R9/RT train/val/test data

In [ ]:
# import
episcan_train_test_dir=DATA_DIR / "Training_Datasets" / "Processed" / "EpiScan"
episcan_r9_train_set=pd.read_csv(f"{episcan_train_test_dir}/episcan_r9_train_set_remaining.csv",index_col=0)
episcan_rt_train_set=pd.read_csv(f"{episcan_train_test_dir}/episcan_rt_train_set_remaining.csv",index_col=0)
episcan_r9_val_set=pd.read_csv(f"{episcan_train_test_dir}/episcan_r9_val_set.csv",index_col=0)
episcan_rt_val_set=pd.read_csv(f"{episcan_train_test_dir}/episcan_rt_val_set.csv",index_col=0)
episcan_r9_test_set=pd.read_csv(f"{episcan_train_test_dir}/episcan_r9_test_set.csv",index_col=0)
episcan_rt_test_set=pd.read_csv(f"{episcan_train_test_dir}/episcan_rt_test_set.csv",index_col=0)
sd3_formatted=pd.read_csv(f"{episcan_train_test_dir}/sd3_formatted.csv",index_col=0)

episcan_r9_val_set["dataset_source"]="r9"
episcan_rt_val_set["dataset_source"]="rt"
episcan_r9_test_set["dataset_source"]="r9"
episcan_rt_test_set["dataset_source"]="rt"

In [ ]:
# process
def processing_wrapper(df):
    df=full_process_data(df,mhc_seq_dict,remove_flank=True,add_pseudo_label=False)
    df=create_seq_allele_column(df)
    df["peptide"]=df.seq.str.replace("$","")
    df["seq_len"]=df.seq.str.replace("$","").apply(len)
    return df

episcan_r9_train_set=processing_wrapper(episcan_r9_train_set)
episcan_rt_train_set=processing_wrapper(episcan_rt_train_set)
episcan_r9_val_set=processing_wrapper(episcan_r9_val_set)
episcan_rt_val_set=processing_wrapper(episcan_rt_val_set)
episcan_r9_test_set=processing_wrapper(episcan_r9_test_set)
episcan_rt_test_set=processing_wrapper(episcan_rt_test_set)
sd3_formatted=processing_wrapper(sd3_formatted)

# merge episcan sets
print("Merging EpiScan R9/RT sets...")
episcan_train=pd.concat([episcan_r9_train_set,episcan_rt_train_set])
episcan_val=pd.concat([episcan_r9_val_set,episcan_rt_val_set])
episcan_test=pd.concat([episcan_r9_test_set,episcan_rt_test_set])
print_data_stats(episcan_train)
print_data_stats(episcan_val)
print_data_stats(episcan_test)
print(end="\n")

Merging EpiScan R9/RT sets...
Total: 2,498,576
Pos: 108,158 | 4.33%
Neg: 2,390,418 | 95.67%
N alleles: 4
Total: 33,800
Pos: 338 | 1.00%
Neg: 33,462 | 99.00%
N alleles: 4
Total: 80,000
Pos: 800 | 1.00%
Neg: 79,200 | 99.00%
N alleles: 4



#### BA train/val/test data

In [ ]:
# import
ba_train_test_dir = DATA_DIR / "Training_Datasets" / "Processed" / "BA"
ba_train=pd.read_csv(f"{ba_train_test_dir}/train_df_ba_for_updating_remaining.csv", index_col=0)
ba_val=pd.read_csv(f"{ba_train_test_dir}//ba_val_set.csv", index_col=0)
ba_test=pd.read_csv(f"{ba_train_test_dir}/test_df_ba_for_updating.csv", index_col=0)

In [ ]:
# process
ba_train=full_process_data(ba_train, mhc_seq_dict, remove_flank=True, add_pseudo_label=False)
ba_train=create_seq_allele_column(ba_train)

ba_val=full_process_data(ba_val, mhc_seq_dict, remove_flank=True, add_pseudo_label=False)
ba_val=create_seq_allele_column(ba_val)

ba_test=full_process_data(ba_test, mhc_seq_dict, remove_flank=True, add_pseudo_label=False)
ba_test=create_seq_allele_column(ba_test)

In [ ]:
# no background, score file for escape (same as all other files)

#### ESCAPE-seq train/val/test data

In [ ]:
# import 
escape_train=pd.read_csv(f"{DATA_DIR}/Training_Datasets/Processed/ESCAPE_seq/escape_con_train_set_remaining_cleaned.csv", index_col=0)
escape_val=pd.read_csv(f"{DATA_DIR}/Training_Datasets/Processed/ESCAPE_seq/escape_con_val_set_cleaned.csv", index_col=0)
escape_test=pd.read_csv(f"{DATA_DIR}/Training_Datasets/Processed/ESCAPE_seq/escape_con_test_set_cleaned.csv", index_col=0)

In [ ]:
# process
escape_train=full_process_data(escape_train, mhc_seq_dict, remove_flank=True, add_pseudo_label=False)
escape_train=create_seq_allele_column(escape_train)

escape_val=full_process_data(escape_val, mhc_seq_dict, remove_flank=True, add_pseudo_label=False)
escape_val=create_seq_allele_column(escape_val)

escape_test=full_process_data(escape_test, mhc_seq_dict, remove_flank=True, add_pseudo_label=False)
escape_test=create_seq_allele_column(escape_test)

In [ ]:
%%time
# rapid save
chp_background_10k_per_allele_x_len_processed_escape_opt = pd.read_pickle(f"{DATA_DIR}/chp_background_10k_per_allele_x_len_processed_escape_opt.pkl")

CPU times: user 4.56 s, sys: 10.5 s, total: 15 s
Wall time: 27.5 s


#### IEDB T cell assay data

In [ ]:
%%time
# import 
iedb_t_cell_data=pd.read_excel(f"{DATA_DIR}/Training_Datasets/Collected/Immunogenicity_Data/IEDB_T_cell_pos_neg_assays_MHC_class_I.xlsx")

# map columns
iedb_column_map=dict(zip(iedb_t_cell_data.columns.tolist(), iedb_t_cell_data.iloc[0].tolist()))

In [ ]:
# isolate and rename columns
iedb_t_cell_data_f=iedb_t_cell_data[["Epitope.2", "Assay.5", "MHC Restriction", "MHC Restriction.4", "Epitope.16", "Epitope.14", "Assay.1", "Epitope.4", "Epitope.5", "Reference.2"]]
iedb_t_cell_data_f.columns=['seq', 'assay_classification','allele','hla_type','species','source_organism','method','modified_residues','modifications','pmid']
iedb_t_cell_data_f.drop([0], inplace=True)
# isolate hla type
iedb_t_cell_data_f=iedb_t_cell_data_f.query("hla_type=='I'")

In [ ]:
# filter out non-canonical seqs
iedb_t_cell_data_f=filter_canonical_sequences(iedb_t_cell_data_f, "seq", amino_acids='ACDEFGHIKLMNPQRSTVWY')

# get all positives (regardless of their alleles) to filter out improperly generated negatives
iedb_t_cell_data_f_all_positive_sequences=iedb_t_cell_data_f.query("assay_classification!='Negative'").seq.tolist()

# standardize allele format
iedb_t_cell_data_f=standardize_allele_format(iedb_t_cell_data_f, no_hla_prefix=False)

# filter sequence length as well
iedb_t_cell_data_f["seq_len"]=iedb_t_cell_data_f.seq.apply(len)
iedb_t_cell_data_f=iedb_t_cell_data_f.query("seq_len >= 8 and seq_len <= 14")

# create label columns
iedb_t_cell_data_f["dataset"]=["IEDB"]*len(iedb_t_cell_data_f)
iedb_t_cell_data_f["label"]=iedb_t_cell_data_f.assay_classification.replace({"Positive":1, "Positive-High":1, "Positive-Intermediate":1, "Positive-Low":1, "Negative":0})

# reset index
iedb_t_cell_data_f.reset_index(drop=True, inplace=True)

# convert pmid to int and fill missing values w/ 0
iedb_t_cell_data_f.pmid=iedb_t_cell_data_f.pmid.fillna(0).astype(int)

In [ ]:
# process
iedb_t_cell_data_f=full_process_data(iedb_t_cell_data_f, mhc_seq_dict, remove_flank=True, add_pseudo_label=False)
iedb_t_cell_data_f=create_seq_allele_column(iedb_t_cell_data_f)
print_data_stats(iedb_t_cell_data_f)

In [ ]:
# clean from ms data sources
seq_allele_overlap=df_col_set_intersection(ms_train_data, iedb_t_cell_data_f, "seq_allele", print_output=True, return_output=True)
iedb_t_cell_data_f_cleaned=iedb_t_cell_data_f[~iedb_t_cell_data_f.seq_allele.isin(seq_allele_overlap)]

seq_allele_overlap=df_col_set_intersection(ms_val_data, iedb_t_cell_data_f_cleaned, "seq_allele", print_output=True, return_output=True)
iedb_t_cell_data_f_cleaned=iedb_t_cell_data_f_cleaned[~iedb_t_cell_data_f_cleaned.seq_allele.isin(seq_allele_overlap)]

seq_allele_overlap=df_col_set_intersection(ms_test_data, iedb_t_cell_data_f_cleaned, "seq_allele", print_output=True, return_output=True)
iedb_t_cell_data_f_cleaned=iedb_t_cell_data_f_cleaned[~iedb_t_cell_data_f_cleaned.seq_allele.isin(seq_allele_overlap)]

print_data_stats(iedb_t_cell_data_f_cleaned)

In [ ]:
# exclude invalid alleles as well
iedb_t_cell_data_f_cleaned=iedb_t_cell_data_f_cleaned[iedb_t_cell_data_f_cleaned.allele.isin(list(mhc_seq_dict.keys()))]
print_data_stats(iedb_t_cell_data_f_cleaned)

In [ ]:
# yd set cleaning
iedb_t_cell_data_f_cleaned_holec=iedb_t_cell_data_f_cleaned.copy()
seq_allele_overlap=df_col_set_intersection(holec_train_set, iedb_t_cell_data_f_cleaned_holec, "seq_allele", print_output=True, return_output=True)
iedb_t_cell_data_f_cleaned_holec=iedb_t_cell_data_f_cleaned_holec[~iedb_t_cell_data_f_cleaned_holec.seq_allele.isin(seq_allele_overlap)]

seq_allele_overlap=df_col_set_intersection(holec_val_set, iedb_t_cell_data_f_cleaned_holec, "seq_allele", print_output=True, return_output=True)
iedb_t_cell_data_f_cleaned_holec=iedb_t_cell_data_f_cleaned_holec[~iedb_t_cell_data_f_cleaned_holec.seq_allele.isin(seq_allele_overlap)]

seq_allele_overlap=df_col_set_intersection(holec_test_set, iedb_t_cell_data_f_cleaned_holec, "seq_allele", print_output=True, return_output=True)
iedb_t_cell_data_f_cleaned_holec=iedb_t_cell_data_f_cleaned_holec[~iedb_t_cell_data_f_cleaned_holec.seq_allele.isin(seq_allele_overlap)]

print_data_stats(iedb_t_cell_data_f_cleaned_holec)

In [ ]:
# episcan set cleaning
iedb_t_cell_data_f_cleaned_episcan=iedb_t_cell_data_f_cleaned.copy()
seq_allele_overlap=df_col_set_intersection(episcan_train, iedb_t_cell_data_f_cleaned_episcan, "seq_allele", print_output=True, return_output=True)
iedb_t_cell_data_f_cleaned_episcan=iedb_t_cell_data_f_cleaned_episcan[~iedb_t_cell_data_f_cleaned_episcan.seq_allele.isin(seq_allele_overlap)]

seq_allele_overlap=df_col_set_intersection(episcan_val, iedb_t_cell_data_f_cleaned_episcan, "seq_allele", print_output=True, return_output=True)
iedb_t_cell_data_f_cleaned_episcan=iedb_t_cell_data_f_cleaned_episcan[~iedb_t_cell_data_f_cleaned_episcan.seq_allele.isin(seq_allele_overlap)]

seq_allele_overlap=df_col_set_intersection(episcan_test, iedb_t_cell_data_f_cleaned_episcan, "seq_allele", print_output=True, return_output=True)
iedb_t_cell_data_f_cleaned_episcan=iedb_t_cell_data_f_cleaned_episcan[~iedb_t_cell_data_f_cleaned_episcan.seq_allele.isin(seq_allele_overlap)]

print_data_stats(iedb_t_cell_data_f_cleaned_episcan)

In [ ]:
# ba set cleaning
iedb_t_cell_data_f_cleaned_ba=iedb_t_cell_data_f_cleaned.copy()
seq_allele_overlap=df_col_set_intersection(ba_train, iedb_t_cell_data_f_cleaned_ba, "seq_allele", print_output=True, return_output=True)
iedb_t_cell_data_f_cleaned_ba=iedb_t_cell_data_f_cleaned_ba[~iedb_t_cell_data_f_cleaned_ba.seq_allele.isin(seq_allele_overlap)]

seq_allele_overlap=df_col_set_intersection(ba_val, iedb_t_cell_data_f_cleaned_ba, "seq_allele", print_output=True, return_output=True)
iedb_t_cell_data_f_cleaned_ba=iedb_t_cell_data_f_cleaned_ba[~iedb_t_cell_data_f_cleaned_ba.seq_allele.isin(seq_allele_overlap)]

seq_allele_overlap=df_col_set_intersection(ba_test, iedb_t_cell_data_f_cleaned_ba, "seq_allele", print_output=True, return_output=True)
iedb_t_cell_data_f_cleaned_ba=iedb_t_cell_data_f_cleaned_ba[~iedb_t_cell_data_f_cleaned_ba.seq_allele.isin(seq_allele_overlap)]

print_data_stats(iedb_t_cell_data_f_cleaned_ba)

In [ ]:
# escape set cleaning
iedb_t_cell_data_f_cleaned_escape=iedb_t_cell_data_f_cleaned.copy()
seq_allele_overlap=df_col_set_intersection(escape_train, iedb_t_cell_data_f_cleaned_escape, "seq_allele", print_output=True, return_output=True)
iedb_t_cell_data_f_cleaned_escape=iedb_t_cell_data_f_cleaned_escape[~iedb_t_cell_data_f_cleaned_escape.seq_allele.isin(seq_allele_overlap)]

seq_allele_overlap=df_col_set_intersection(escape_val, iedb_t_cell_data_f_cleaned_escape, "seq_allele", print_output=True, return_output=True)
iedb_t_cell_data_f_cleaned_escape=iedb_t_cell_data_f_cleaned_escape[~iedb_t_cell_data_f_cleaned_escape.seq_allele.isin(seq_allele_overlap)]

seq_allele_overlap=df_col_set_intersection(escape_test, iedb_t_cell_data_f_cleaned_escape, "seq_allele", print_output=True, return_output=True)
iedb_t_cell_data_f_cleaned_escape=iedb_t_cell_data_f_cleaned_escape[~iedb_t_cell_data_f_cleaned_escape.seq_allele.isin(seq_allele_overlap)]

print_data_stats(iedb_t_cell_data_f_cleaned_escape)

### Load models

 - YD and ESCAPE-seq models already scored our background set, however, EpiScan and BA models have not and will need to do so.

In [ ]:
# setup tokenizer
tokenizer = AminoAcidTokenizer()

In [ ]:
# set base model params
base_model_params={
        'vocab_size': tokenizer.vocab_size,
        'embed_dim': 480,
        'fc_hidden_dims': [240, 120], 
        'fc_dropout': 0.1,
        'n_heads': 16,
        'max_peptide_len': 34,
        'max_mhc_len': 34,
        "n_peptide_layers": 2,
        "n_mhc_layers": 1,
        'n_concat_layers': 3,
        'add_pos_enc': True,
        'use_segment_embedder': True,
        'dim_feedforward_multiplier': 1,
        'debugging': False,
        'fc_output_dim': 1,
        'pooling': 'cls', # cls, mean, min, max, attn, for sequence ffnn head
        "ffnn_use_layernorm": False, 
        'use_feature_table': True,
        "feature_table": processed_feature_table,
        "sequence_embedder_feature_names": feature_names,
        'sequence_embedder_feature_mode': "concat",  # replace or concat
        'sequence_embedder_tokenizer': tokenizer,
        "peptide_start_pos": 1, 
        "mhc_start_pos": 35, 
        "pos_enc_type": "sincos", 
        "seed": 42,

        # peptide conformer
        "add_peptide_conformer": True, 
        "peptide_conformer_num_layers": 2,
        "peptide_conformer_n_heads": 16,
        "peptide_conformer_conv_kernel": 5,
        "peptide_conformer_dropout": 0.0,

        # mhc conformer
        "add_mhc_conformer": False, 
        "mhc_conformer_num_layers": 2,
        "mhc_conformer_n_heads": 16,
        "mhc_conformer_conv_kernel": 5,
        "mhc_conformer_dropout": 0.0,

        # fusion conformer parameters
        "add_fusion_conformer": True, 
        "fusion_conformer_num_layers": 1,
        "fusion_conformer_n_heads": 16,
        "fusion_conformer_conv_kernel": 7,
        "fusion_conformer_dropout": 0.0,
        "fusion_conformer_mask_bos": True,
        "ff_pre_post_swiglu": False,

        # legacy init
        "legacy_init_compat": True,
}

In [ ]:
%%time
# load base model (optionally, can load model_params directly)
# run_name = "MHCPrime_Base"
# run_dir = MODEL_CHECKPOINTS_DIR / run_name
# model_dir = run_dir / "Checkpoints"

# mhcprime=init_model(base_model_params, print_check=False, print_params=True)
# mhcprime.load_state_dict(torch.load(f"{model_dir}/model_final.pt"))
# mhcprime.eval()
# print("Model loaded from checkpoint...")
# clear_all_gpu_memory()

base_model_dir = MODEL_CHECKPOINTS_DIR / "MHCPrime_Base_for_Ablations" / "Checkpoints"
ckpt_path = base_model_dir / "model_final.pt"

device = torch.device(f"cuda:{GPU_ID}" if torch.cuda.is_available() else "cpu")

clear_all_gpu_memory()

mhcprime = init_model(
    base_model_params,
    print_check=False,
    print_params=True,
)

state_dict = torch.load(
    ckpt_path,
    map_location="cpu",
)

mhcprime.load_state_dict(state_dict, strict=True)
mhcprime.to(device)
mhcprime.eval()

print("Model loaded from checkpoint...")
print(f"Checkpoint: {ckpt_path}")
print(f"Model device: {next(mhcprime.parameters()).device}")

print(end="\n")


In [ ]:
# get yd models for scoring
base_model=copy.deepcopy(mhcprime)

base_and_ft_models={
    # "mhcprime_abt": base_model, 
}

for model_file in os.listdir(f"{MODEL_CHECKPOINTS_DIR}/YD_Validation_Selected_Models_5_Seeds"):
    model_name=model_file.removesuffix(".pt")
    print(model_name)

    ckpt_path = f"{MODEL_CHECKPOINTS_DIR}/YD_Validation_Selected_Models_5_Seeds/{model_file}"

    device = torch.device(f"cuda:{GPU_ID}" if torch.cuda.is_available() else "cpu")

    clear_all_gpu_memory()

    model = init_model(
        base_model_params,
        print_check=False,
        print_params=True,
    )

    state_dict = torch.load(
        ckpt_path,
        map_location="cpu",
    )

    model.load_state_dict(state_dict, strict=True)
    model.to(device)
    model.eval()

    print("Model loaded from checkpoint...")
    print(f"Checkpoint: {ckpt_path}")
    print(f"Model device: {next(model.parameters()).device}")

    base_and_ft_models[model_name]=model

datasplits=[1,2,3,4]
for ds in datasplits:
    ds_seed=ds+4
    model_path=f"{MODEL_CHECKPOINTS_DIR}/YD_Validation_Selected_Models_CV_Datasplits/CV_Datasplit_{ds}"
    for model_file in os.listdir(model_path):
        model_name=model_file.removesuffix(".pt")
        print(model_name)
        model_name=model_name.replace("seed_0", f"seed_{ds_seed}")
        print(model_name)

        ckpt_path = f"{MODEL_CHECKPOINTS_DIR}/YD_Validation_Selected_Models_CV_Datasplits/CV_Datasplit_{ds}/{model_file}"

        device = torch.device(f"cuda:{GPU_ID}" if torch.cuda.is_available() else "cpu")

        clear_all_gpu_memory()

        model = init_model(
            base_model_params,
            print_check=False,
            print_params=True,
        )

        state_dict = torch.load(
            ckpt_path,
            map_location="cpu",
        )

        model.load_state_dict(state_dict, strict=True)
        model.to(device)
        model.eval()

        print("Model loaded from checkpoint...")
        print(f"Checkpoint: {ckpt_path}")
        print(f"Model device: {next(model.parameters()).device}")

        base_and_ft_models[model_name]=model

print(len(base_and_ft_models), base_and_ft_models.keys())

In [ ]:
# episcan models
base_model=copy.deepcopy(mhcprime)

base_and_ft_models={
    "mhcprime_abt": base_model, 
}

for model_file in os.listdir(f"{MODEL_CHECKPOINTS_DIR}/EpiScan_R9_RT_Validation_Selected_Models_5_Seeds"):
    model_name=model_file.removesuffix(".pt")
    print(model_name)

    ckpt_path = f"{MODEL_CHECKPOINTS_DIR}/EpiScan_R9_RT_Validation_Selected_Models_5_Seeds/{model_file}"

    device = torch.device(f"cuda:{GPU_ID}" if torch.cuda.is_available() else "cpu")

    clear_all_gpu_memory()

    model = init_model(
        base_model_params,
        print_check=False,
        print_params=True,
    )

    state_dict = torch.load(
        ckpt_path,
        map_location="cpu",
    )

    model.load_state_dict(state_dict, strict=True)
    model.to(device)
    model.eval()

    print("Model loaded from checkpoint...")
    print(f"Checkpoint: {ckpt_path}")
    print(f"Model device: {next(model.parameters()).device}")

    base_and_ft_models[model_name]=model

for model_file in os.listdir(f"{MODEL_CHECKPOINTS_DIR}/EpiScan_R9_RT_Validation_Selected_Models_5_Seeds_FT_PepCL_Optimal"):
    model_name=model_file.removesuffix(".pt")
    print(model_name)

    ckpt_path = f"{MODEL_CHECKPOINTS_DIR}/EpiScan_R9_RT_Validation_Selected_Models_5_Seeds_FT_PepCL_Optimal/{model_file}"

    device = torch.device(f"cuda:{GPU_ID}" if torch.cuda.is_available() else "cpu")

    clear_all_gpu_memory()

    model = init_model(
        base_model_params,
        print_check=False,
        print_params=True,
    )

    state_dict = torch.load(
        ckpt_path,
        map_location="cpu",
    )

    model.load_state_dict(state_dict, strict=True)
    model.to(device)
    model.eval()

    print("Model loaded from checkpoint...")
    print(f"Checkpoint: {ckpt_path}")
    print(f"Model device: {next(model.parameters()).device}")

    base_and_ft_models[model_name]=model


datasplits=[1,2,3,4]
for ds in datasplits:
    ds_seed=ds+4
    model_path=f"{MODEL_CHECKPOINTS_DIR}/EpiScan_R9_RT_Validation_Selected_Models_CV_Datasplits/CV_Datasplit_{ds}"
    for model_file in os.listdir(model_path):
        model_name=model_file.removesuffix(".pt")
        print(model_name)
        model_name=model_name.replace("seed_0", f"seed_{ds_seed}")
        print(model_name)

        ckpt_path = f"{MODEL_CHECKPOINTS_DIR}/EpiScan_R9_RT_Validation_Selected_Models_CV_Datasplits/CV_Datasplit_{ds}/{model_file}"

        device = torch.device(f"cuda:{GPU_ID}" if torch.cuda.is_available() else "cpu")

        clear_all_gpu_memory()

        model = init_model(
            base_model_params,
            print_check=False,
            print_params=True,
        )

        state_dict = torch.load(
            ckpt_path,
            map_location="cpu",
        )

        model.load_state_dict(state_dict, strict=True)
        model.to(device)
        model.eval()

        print("Model loaded from checkpoint...")
        print(f"Checkpoint: {ckpt_path}")
        print(f"Model device: {next(model.parameters()).device}")

        base_and_ft_models[model_name]=model

for ds in datasplits:
    ds_seed=ds+4
    model_path=f"{MODEL_CHECKPOINTS_DIR}/EpiScan_R9_RT_Validation_Selected_Models_CV_Datasplits_FT_PepCL_Optimal/CV_Datasplit_{ds}"
    for model_file in os.listdir(model_path):
        model_name=model_file.removesuffix(".pt")
        print(model_name)
        model_name=model_name.replace("seed_0", f"seed_{ds_seed}")
        print(model_name)

        ckpt_path = f"{MODEL_CHECKPOINTS_DIR}/EpiScan_R9_RT_Validation_Selected_Models_CV_Datasplits_FT_PepCL_Optimal/CV_Datasplit_{ds}/{model_file}"

        device = torch.device(f"cuda:{GPU_ID}" if torch.cuda.is_available() else "cpu")

        clear_all_gpu_memory()

        model = init_model(
            base_model_params,
            print_check=False,
            print_params=True,
        )

        state_dict = torch.load(
            ckpt_path,
            map_location="cpu",
        )

        model.load_state_dict(state_dict, strict=True)
        model.to(device)
        model.eval()

        print("Model loaded from checkpoint...")
        print(f"Checkpoint: {ckpt_path}")
        print(f"Model device: {next(model.parameters()).device}")

        base_and_ft_models[model_name]=model

print(len(base_and_ft_models), base_and_ft_models.keys())
# get models for scoring

for model_file in os.listdir(f"{MODEL_CHECKPOINTS_DIR}/EpiScan_R9_Validation_Selected_Models_5_Seeds"):
    model_name=model_file.removesuffix(".pt")
    print(model_name)

    ckpt_path = f"{MODEL_CHECKPOINTS_DIR}/EpiScan_R9_Validation_Selected_Models_5_Seeds/{model_file}"

    device = torch.device(f"cuda:{GPU_ID}" if torch.cuda.is_available() else "cpu")

    clear_all_gpu_memory()

    model = init_model(
        base_model_params,
        print_check=False,
        print_params=True,
    )

    state_dict = torch.load(
        ckpt_path,
        map_location="cpu",
    )

    model.load_state_dict(state_dict, strict=True)
    model.to(device)
    model.eval()

    print("Model loaded from checkpoint...")
    print(f"Checkpoint: {ckpt_path}")
    print(f"Model device: {next(model.parameters()).device}")

    base_and_ft_models[model_name]=model

for model_file in os.listdir(f"{MODEL_CHECKPOINTS_DIR}/EpiScan_R9_Validation_Selected_Models_5_Seeds_FT_Final"):
    model_name=model_file.removesuffix(".pt")
    print(model_name)

    ckpt_path = f"{MODEL_CHECKPOINTS_DIR}/EpiScan_R9_Validation_Selected_Models_5_Seeds_FT_Final/{model_file}"

    device = torch.device(f"cuda:{GPU_ID}" if torch.cuda.is_available() else "cpu")

    clear_all_gpu_memory()

    model = init_model(
        base_model_params,
        print_check=False,
        print_params=True,
    )

    state_dict = torch.load(
        ckpt_path,
        map_location="cpu",
    )

    model.load_state_dict(state_dict, strict=True)
    model.to(device)
    model.eval()

    print("Model loaded from checkpoint...")
    print(f"Checkpoint: {ckpt_path}")
    print(f"Model device: {next(model.parameters()).device}")

    base_and_ft_models[model_name]=model


datasplits=[1,2,3,4]
for ds in datasplits:
    ds_seed=ds+4
    model_path=f"{MODEL_CHECKPOINTS_DIR}/EpiScan_R9_Validation_Selected_Models_CV_Datasplits/CV_Datasplit_{ds}"
    for model_file in os.listdir(model_path):
        model_name=model_file.removesuffix(".pt")
        print(model_name)
        model_name=model_name.replace("seed_0", f"seed_{ds_seed}")
        print(model_name)

        ckpt_path = f"{MODEL_CHECKPOINTS_DIR}/EpiScan_R9_Validation_Selected_Models_CV_Datasplits/CV_Datasplit_{ds}/{model_file}"

        device = torch.device(f"cuda:{GPU_ID}" if torch.cuda.is_available() else "cpu")

        clear_all_gpu_memory()

        model = init_model(
            base_model_params,
            print_check=False,
            print_params=True,
        )

        state_dict = torch.load(
            ckpt_path,
            map_location="cpu",
        )

        model.load_state_dict(state_dict, strict=True)
        model.to(device)
        model.eval()

        print("Model loaded from checkpoint...")
        print(f"Checkpoint: {ckpt_path}")
        print(f"Model device: {next(model.parameters()).device}")

        base_and_ft_models[model_name]=model

for ds in datasplits:
    ds_seed=ds+4
    model_path=f"{MODEL_CHECKPOINTS_DIR}/EpiScan_R9_Validation_Selected_Models_CV_Datasplits_FT_Final/CV_Datasplit_{ds}"
    for model_file in os.listdir(model_path):
        model_name=model_file.removesuffix(".pt")
        print(model_name)
        model_name=model_name.replace("seed_0", f"seed_{ds_seed}")
        print(model_name)

        ckpt_path = f"{MODEL_CHECKPOINTS_DIR}/EpiScan_R9_Validation_Selected_Models_CV_Datasplits_FT_Final/CV_Datasplit_{ds}/{model_file}"

        device = torch.device(f"cuda:{GPU_ID}" if torch.cuda.is_available() else "cpu")

        clear_all_gpu_memory()

        model = init_model(
            base_model_params,
            print_check=False,
            print_params=True,
        )

        state_dict = torch.load(
            ckpt_path,
            map_location="cpu",
        )

        model.load_state_dict(state_dict, strict=True)
        model.to(device)
        model.eval()

        print("Model loaded from checkpoint...")
        print(f"Checkpoint: {ckpt_path}")
        print(f"Model device: {next(model.parameters()).device}")

        base_and_ft_models[model_name]=model

print(len(base_and_ft_models), base_and_ft_models.keys())

In [ ]:
# prefixes
def filter_dict_by_prefixes(d, prefixes):
    return {k: v for k, v in d.items() if any(k.startswith(p) for p in prefixes)}

r9_models=["episcan_r9_ft_steps_300_lr_2eN4", "episcan_r9_pepcl_r_dr_steps_300_lr_2eN4", "episcan_r9_pepcl_r_dr_steps_800_lr_2eN4"]

r9rt_models=["episcan_r9rt_ft_steps_400_lr_2eN4", "episcan_r9rt_pepcl_r_dr_steps_300_lr_2eN4", "episcan_r9rt_pepcl_r_dr_steps_400_lr_2eN4"]

episcan_model_prefixes=r9_models + r9rt_models
base_and_ft_models=filter_dict_by_prefixes(base_and_ft_models, episcan_model_prefixes)
print(len(base_and_ft_models), base_and_ft_models.keys())

In [ ]:
# ba con models
base_model=copy.deepcopy(mhcprime)

base_and_ft_models={
    # "mhcprime_abt_rapid": base_model, 
}

for model_file in os.listdir(f"{MODEL_CHECKPOINTS_DIR}/BA_CON_Validation_Selected_Models_5_Seeds"):
    model_name=model_file.removesuffix(".pt")
    print(model_name)

    ckpt_path = f"{MODEL_CHECKPOINTS_DIR}/BA_CON_Validation_Selected_Models_5_Seeds/{model_file}"

    device = torch.device(f"cuda:{GPU_ID}" if torch.cuda.is_available() else "cpu")

    clear_all_gpu_memory()

    model = init_model(
        base_model_params,
        print_check=False,
        print_params=True,
    )

    state_dict = torch.load(
        ckpt_path,
        map_location="cpu",
    )

    model.load_state_dict(state_dict, strict=True)
    model.to(device)
    model.eval()

    print("Model loaded from checkpoint...")
    print(f"Checkpoint: {ckpt_path}")
    print(f"Model device: {next(model.parameters()).device}")

    base_and_ft_models[model_name]=model

datasplits=[1,2,3,4]
for ds in datasplits:
    ds_seed=ds+4
    model_path=f"{MODEL_CHECKPOINTS_DIR}/BA_CON_Validation_Selected_Models_CV_Datasplits/CV_Datasplit_{ds}"
    for model_file in os.listdir(model_path):
        model_name=model_file.removesuffix(".pt")
        print(model_name)
        model_name=model_name.replace("seed_0", f"seed_{ds_seed}")
        print(model_name)

        ckpt_path = f"{MODEL_CHECKPOINTS_DIR}/BA_CON_Validation_Selected_Models_CV_Datasplits/CV_Datasplit_{ds}/{model_file}"

        device = torch.device(f"cuda:{GPU_ID}" if torch.cuda.is_available() else "cpu")

        clear_all_gpu_memory()

        model = init_model(
            base_model_params,
            print_check=False,
            print_params=True,
        )

        state_dict = torch.load(
            ckpt_path,
            map_location="cpu",
        )

        model.load_state_dict(state_dict, strict=True)
        model.to(device)
        model.eval()

        print("Model loaded from checkpoint...")
        print(f"Checkpoint: {ckpt_path}")
        print(f"Model device: {next(model.parameters()).device}")

        base_and_ft_models[model_name]=model

print(len(base_and_ft_models), base_and_ft_models.keys())

In [ ]:
# get models for scoring
base_model=copy.deepcopy(mhcprime)

base_and_ft_models={
    "mhcprime_abt_rapid": base_model, 
}

for model_file in os.listdir(f"{MODEL_CHECKPOINTS_DIR}/ESCAPE_seq_CON_Validation_Selected_Models_5_Seeds_Cleaned"):
    model_name=model_file.removesuffix(".pt")
    print(model_name)

    ckpt_path = f"{MODEL_CHECKPOINTS_DIR}/ESCAPE_seq_CON_Validation_Selected_Models_5_Seeds_Cleaned/{model_file}"

    device = torch.device(f"cuda:{GPU_ID}" if torch.cuda.is_available() else "cpu")

    clear_all_gpu_memory()

    model = init_model(
        base_model_params,
        print_check=False,
        print_params=True,
    )

    state_dict = torch.load(
        ckpt_path,
        map_location="cpu",
    )

    model.load_state_dict(state_dict, strict=True)
    model.to(device)
    model.eval()

    print("Model loaded from checkpoint...")
    print(f"Checkpoint: {ckpt_path}")
    print(f"Model device: {next(model.parameters()).device}")

    base_and_ft_models[model_name]=model

datasplits=[1,2,3,4]
for ds in datasplits:
    ds_seed=ds+4
    model_path=f"{MODEL_CHECKPOINTS_DIR}/ESCAPE_seq_CON_Validation_Selected_Models_CV_Datasplits_Cleaned/CV_Datasplit_{ds}"
    for model_file in os.listdir(model_path):
        model_name=model_file.removesuffix(".pt")
        print(model_name)
        model_name=model_name.replace("seed_0", f"seed_{ds_seed}")
        print(model_name)

        ckpt_path = f"{MODEL_CHECKPOINTS_DIR}/ESCAPE_seq_CON_Validation_Selected_Models_CV_Datasplits_Cleaned/CV_Datasplit_{ds}/{model_file}"

        device = torch.device(f"cuda:{GPU_ID}" if torch.cuda.is_available() else "cpu")

        clear_all_gpu_memory()

        model = init_model(
            base_model_params,
            print_check=False,
            print_params=True,
        )

        state_dict = torch.load(
            ckpt_path,
            map_location="cpu",
        )

        model.load_state_dict(state_dict, strict=True)
        model.to(device)
        model.eval()

        print("Model loaded from checkpoint...")
        print(f"Checkpoint: {ckpt_path}")
        print(f"Model device: {next(model.parameters()).device}")

        base_and_ft_models[model_name]=model

print(len(base_and_ft_models), base_and_ft_models.keys())

### Inference

In [ ]:
del base_and_ft_models["mhcprime_abt_rapid"]

In [ ]:
%%time
"""
Run so far:

 - escape_con models
 - yd models (under yd file name for background)
 - ba_con models
 - episcan r9 models
 - episcan r9/rt models

"""
%%time
"""
Score background
"""
# chp background
data_to_run_path=f"{DATA_DIR}/chp_background_10k_per_allele_x_len_processed_escape_opt.pt"

# run_bg=False
run_bg=True

if run_bg:
    for model_name, model in base_and_ft_models.items():
        print(model_name)
        model.eval()
        with torch.no_grad():
            num_workers=16
            output_scores=run_mhcprime_fast(model, data_to_run_path, batch_size=3072, num_workers=num_workers, compile_model=False, use_domain_id=False)

        chp_background_10k_per_allele_x_len_processed_escape_opt[model_name]=output_scores.detach().cpu().tolist() 

In [ ]:
%%time
# save background set as pkl file
# rapid save
# chp_background_10k_per_allele_x_len_processed_escape_opt.to_pickle(
#     f"{DATA_DIR}/chp_background_10k_per_allele_x_len_processed_escape_opt.pkl",
#     protocol=5,
#     compression=None,
# )

#### YD models

In [ ]:
# add back
base_model=copy.deepcopy(mhcprime)
base_and_ft_models["mhcprime_abt"]=base_model

In [ ]:
%%time
# iedb_t_cell_data_f_cleaned_holec
build_caches=True
# build_caches=False

if build_caches:
    data_to_run_path=f"{DATA_DIR}/dataset_cache.pt"

    build_token_cache(iedb_t_cell_data_f_cleaned_holec, data_to_run_path, tokenizer, use_domain_id=False)

data_to_run_path=f"{DATA_DIR}/dataset_cache.pt"
for model_name, model in base_and_ft_models.items():
    model.eval()
    with torch.no_grad():
        output_scores=run_mhcprime_fast(model, data_to_run_path, batch_size=3072, num_workers=16, compile_model=False, use_domain_id=False)

    iedb_t_cell_data_f_cleaned_holec[model_name]=output_scores.detach().cpu().tolist()

In [ ]:
# escape con and cls rank values for cleaned iedb
iedb_t_cell_data_f_cleaned_holec["seq_len"]=iedb_t_cell_data_f_cleaned_holec.seq.str.replace("$", "").apply(len)

score_cols=list(base_and_ft_models.keys())
print(score_cols)

# all the same model, just named differently
# score_cols+=["mhcprime"]
# score_cols+=["mhcprime_abt"]
# score_cols+=["mhcprime_abt_rapid"]

iedb_t_cell_data_f_cleaned_holec = add_background_percentile_ranks(
    test_df=iedb_t_cell_data_f_cleaned_holec,
    background_df=chp_background_10k_per_allele_x_len_processed_yd_opt, # for yd models
    allele_col="allele",
    seq_len_col="seq_len",
    score_cols=score_cols,
    rank_mode="global",
    suffix="_rank_r_chp",
    reverse=True,
    copy=True,
)

In [ ]:
# save
# iedb_t_cell_data_f_cleaned_holec.to_csv(f"{DATA_DIR}/Training_Datasets/Collected/Immunogenicity_Data/iedb_t_cell_data_f_cleaned_holec.csv")

#### EpiScan models

In [ ]:
# add back
base_model=copy.deepcopy(mhcprime)
base_and_ft_models["mhcprime_abt_rapid"]=base_model

In [ ]:
%%time
build_caches=True
# build_caches=False

if build_caches:
    data_to_run_path=f"{DATA_DIR}/dataset_cache.pt"

    build_token_cache(iedb_t_cell_data_f_cleaned_episcan, data_to_run_path, tokenizer, use_domain_id=False)

data_to_run_path=f"{DATA_DIR}/dataset_cache.pt"
for model_name, model in base_and_ft_models.items():
    model.eval()
    with torch.no_grad():
        output_scores=run_mhcprime_fast(model, data_to_run_path, batch_size=3072, num_workers=16, compile_model=False, use_domain_id=False)

    iedb_t_cell_data_f_cleaned_episcan[model_name]=output_scores.detach().cpu().tolist()

In [ ]:
# escape con and cls rank values for cleaned iedb
iedb_t_cell_data_f_cleaned_episcan["seq_len"]=iedb_t_cell_data_f_cleaned_episcan.seq.str.replace("$", "").apply(len)

score_cols=list(base_and_ft_models.keys())
print(score_cols)

# all the same model, just named differently
# score_cols+=["mhcprime"]
# score_cols+=["mhcprime_abt"]
# score_cols+=["mhcprime_abt_rapid"]

iedb_t_cell_data_f_cleaned_episcan = add_background_percentile_ranks(
    test_df=iedb_t_cell_data_f_cleaned_episcan,
    background_df=chp_background_10k_per_allele_x_len_processed_escape_opt,
    allele_col="allele",
    seq_len_col="seq_len",
    score_cols=score_cols,
    rank_mode="global",
    suffix="_rank_r_chp",
    reverse=True,
    copy=True,
)

In [ ]:
# save
# iedb_t_cell_data_f_cleaned_episcan.to_csv(f"{DATA_DIR}/Training_Datasets/Collected/Immunogenicity_Data/iedb_t_cell_data_f_cleaned_episcan.csv")

#### BA CON and CLS models

In [ ]:
# add back
base_model=copy.deepcopy(mhcprime)
base_and_ft_models["mhcprime_abt_rapid"]=base_model

In [ ]:
%%time
build_caches=True
# build_caches=False

if build_caches:
    data_to_run_path=f"{DATA_DIR}/dataset_cache.pt"

    build_token_cache(iedb_t_cell_data_f_cleaned_ba, data_to_run_path, tokenizer, use_domain_id=False)

data_to_run_path=f"{DATA_DIR}/dataset_cache.pt"
for model_name, model in base_and_ft_models.items():
    model.eval()
    with torch.no_grad():
        output_scores=run_mhcprime_fast(model, data_to_run_path, batch_size=3072, num_workers=16, compile_model=False, use_domain_id=False)

    iedb_t_cell_data_f_cleaned_ba[model_name]=output_scores.detach().cpu().tolist()

In [ ]:
# get rank values
iedb_t_cell_data_f_cleaned_ba["seq_len"]=iedb_t_cell_data_f_cleaned_ba.seq.str.replace("$", "").apply(len)

score_cols=list(base_and_ft_models.keys())
print(score_cols)

# all the same model, just named differently
# score_cols+=["mhcprime"]
# score_cols+=["mhcprime_abt"]
# score_cols+=["mhcprime_abt_rapid"]

iedb_t_cell_data_f_cleaned_ba = add_background_percentile_ranks(
    test_df=iedb_t_cell_data_f_cleaned_ba,
    background_df=chp_background_10k_per_allele_x_len_processed_escape_opt, # for escape models
    allele_col="allele",
    seq_len_col="seq_len",
    score_cols=score_cols,
    rank_mode="global",
    suffix="_rank_r_chp",
    reverse=True,
    copy=True,
)

In [ ]:
# save
# iedb_t_cell_data_f_cleaned_ba.to_csv(f"{DATA_DIR}/Training_Datasets/Collected/Immunogenicity_Data/iedb_t_cell_data_f_cleaned_ba.csv")

#### ESCAPE-seq CON models

In [ ]:
%%time
# escape con models on cleaned iedb dataset
build_caches=True
# build_caches=False

if build_caches:
    data_to_run_path=f"{DATA_DIR}/dataset_cache.pt"

    build_token_cache(iedb_t_cell_data_f_cleaned_escape, data_to_run_path, tokenizer, use_domain_id=False)

data_to_run_path=f"{DATA_DIR}/dataset_cache.pt"
for model_name, model in base_and_ft_models.items():
    model.eval()
    with torch.no_grad():
        output_scores=run_mhcprime_fast(model, data_to_run_path, batch_size=3072, num_workers=16, compile_model=False, use_domain_id=False)

    iedb_t_cell_data_f_cleaned_escape[model_name]=output_scores.detach().cpu().tolist()

In [ ]:
# escape con and cls rank values for cleaned iedb
iedb_t_cell_data_f_cleaned_escape["seq_len"]=iedb_t_cell_data_f_cleaned_escape.seq.str.replace("$", "").apply(len)

score_cols=list(base_and_ft_models.keys())
print(score_cols)

# all the same model, just named differently
# score_cols+=["mhcprime"]
# score_cols+=["mhcprime_abt"]
# score_cols+=["mhcprime_abt_rapid"]

iedb_t_cell_data_f_cleaned_escape = add_background_percentile_ranks(
    test_df=iedb_t_cell_data_f_cleaned_escape,
    background_df=chp_background_10k_per_allele_x_len_processed_escape_opt, # for escape models
    allele_col="allele",
    seq_len_col="seq_len",
    score_cols=score_cols,
    rank_mode="global",
    suffix="_rank_r_chp",
    reverse=True,
    copy=True,
)

In [ ]:
# save
# iedb_t_cell_data_f_cleaned_escape.to_csv(f"{DATA_DIR}/Training_Datasets/Collected/Immunogenicity_Data/iedb_t_cell_data_f_cleaned_escape.csv")

### Analysis

In [ ]:
def avg_score_per_allele(df, score_cols, allele_col="allele"):
    cols = [c for c in score_cols if c in df.columns]
    return df.groupby(allele_col, sort=False)[cols].mean()

def average_prefix_groups(df, prefix_map, suffix=None, collapse_alleles=False):
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    nonnumeric_cols = [c for c in df.columns if c not in numeric_cols]

    if suffix is not None:
        eligible = [c for c in numeric_cols if c.endswith(suffix)]
    else:
        eligible = list(numeric_cols)

    matched = set()
    score_out = pd.DataFrame(index=df.index)
    for final_name, prefix in prefix_map.items():
        cols = [c for c in eligible if c.startswith(prefix)]
        if cols:
            score_out[final_name] = df[cols].mean(axis=1)
            matched.update(cols)
    for c in eligible:
        if c in matched:
            continue
        name = c[:-len(suffix)] if suffix is not None else c
        score_out[name] = df[c]

    if collapse_alleles:
        return score_out.mean(axis=0).to_frame("avg_score").rename_axis("score_col")

    out = pd.concat([df[nonnumeric_cols], score_out], axis=1)
    return out

def pct_recovered_per_allele(df, score_cols, threshold, allele_col="allele"):
    def frac(g):
        return pd.Series({col: (g[col] >= threshold).sum() / len(g) * 100 for col in score_cols})
    out = df.groupby(allele_col, sort=False).apply(frac)
    out.index.name = allele_col
    return out

# prefix helper
def build_score_cols(prefixes, seeds):
    return [f"{prefix}_seed_{seed}" for prefix in prefixes for seed in seeds]

def filter_alleles_min_samples(df, n, allele_col="allele"):
    counts = df[allele_col].value_counts()
    keep = counts.index[counts >= n]
    return df[df[allele_col].isin(keep)].copy()

def recovery_table_over_thresholds(df, score_cols, thresholds, prefix_map, allele_col="allele", base_col=None, per_allele=True):
    df = df.reset_index(drop=True)
    rec_cols = list(score_cols) if (base_col is None or base_col in score_cols) else list(score_cols) + [base_col]

    def _collapse(s):
        matched, out = set(), {}
        for final_name, prefix in prefix_map.items():
            cols = [c for c in s.index if c.startswith(prefix)]
            if cols:
                out[final_name] = s[cols].mean()
                matched.update(cols)
        for c in s.index:
            if c not in matched:
                out[c] = s[c]
        return pd.Series(out)

    rows = {}
    for t in thresholds:
        if per_allele:
            rec = (df[rec_cols] >= t).groupby(df[allele_col].values).mean() * 100 # rows=alleles, cols=score cols
            if base_col is not None:
                rec = rec.sub(rec[base_col], axis=0) # delta before seed-averaging
            s = rec.mean(axis=0) # average recovery across alleles
        else:
            s = (df[rec_cols] >= t).mean() * 100 # one "allele" = whole dataset
            if base_col is not None:
                s = s - s[base_col]
        rows[t] = _collapse(s) # prefix-average + keep non-prefix cols

    out = pd.DataFrame(rows).T
    out.index.name = "threshold"
    return out

def split_out_overlap(df1, df2, seq_col="seq", allele_col="allele"):
    df1 = df1.reset_index(drop=True); df2 = df2.reset_index(drop=True)
    key1 = list(zip(df1[seq_col], df1[allele_col]))
    key2 = set(zip(df2[seq_col], df2[allele_col]))
    mask1 = pd.Series([k in key2 for k in key1], index=df1.index)
    mask2 = pd.Series([k in set(key1) for k in zip(df2[seq_col], df2[allele_col])], index=df2.index)
    overlap_df = df1[mask1].reset_index(drop=True)
    print(f"Overlap: {overlap_df[[seq_col, allele_col]].drop_duplicates().shape[0]} seq_allele pairs")
    return df1[~mask1].reset_index(drop=True), df2[~mask2].reset_index(drop=True), overlap_df

#### Load data

In [ ]:
# load
iedb_t_cell_data_f_cleaned_holec=pd.read_csv(f"{DATA_DIR}/Training_Datasets/Collected/Immunogenicity_Data/iedb_t_cell_data_f_cleaned_holec.csv", index_col=0)
iedb_t_cell_data_f_cleaned_episcan=pd.read_csv(f"{DATA_DIR}/Training_Datasets/Collected/Immunogenicity_Data/iedb_t_cell_data_f_cleaned_episcan.csv", index_col=0)
iedb_t_cell_data_f_cleaned_ba=pd.read_csv(f"{DATA_DIR}/Training_Datasets/Collected/Immunogenicity_Data/iedb_t_cell_data_f_cleaned_ba.csv", index_col=0)
iedb_t_cell_data_f_cleaned_escape=pd.read_csv(f"{DATA_DIR}/Training_Datasets/Collected/Immunogenicity_Data/iedb_t_cell_data_f_cleaned_escape.csv", index_col=0)

# fix col names
iedb_t_cell_data_f_cleaned_episcan.rename(columns={"mhcprime_abt_rapid": "mhcprime_abt", "mhcprime_abt_rapid_rank_r_chp": "mhcprime_abt_rank_r_chp"}, inplace=True)
iedb_t_cell_data_f_cleaned_ba.rename(columns={"mhcprime_abt_rapid": "mhcprime_abt", "mhcprime_abt_rapid_rank_r_chp": "mhcprime_abt_rank_r_chp"}, inplace=True)
iedb_t_cell_data_f_cleaned_escape.rename(columns={"mhcprime_abt_rapid": "mhcprime_abt", "mhcprime_abt_rapid_rank_r_chp": "mhcprime_abt_rank_r_chp"}, inplace=True)

#### Assay-specific model prefixes

In [ ]:
# get score and rank score cols
dataset_prefix_dict={
    "yd": ["holec_ft_steps_300_lr_2eN4", "holec_pepcl_r_dr_steps_300_lr_2eN4"],
    "episcan_r9": ["episcan_r9_ft_steps_300_lr_2eN4", "episcan_r9_pepcl_r_dr_steps_300_lr_2eN4"],
    "episcan_r9rt": ["episcan_r9rt_ft_steps_400_lr_2eN4", "episcan_r9rt_pepcl_r_dr_steps_300_lr_2eN4"],
    "ba_con": ["ba_con_ft_steps_200_lr_2eN4", "ba_con_pepcl_r_dr_steps_300_lr_2eN4"],
    "escape_con": ["escape_con_ft_steps_200_lr_2eN4", "escape_con_pepcl_r_dr_steps_300_lr_2eN4"],
}

seeds=[0,1,2,3,4,5,6,7,8]

dataset_score_cols={}
dataset_rank_score_cols={}

for d, prefixes in dataset_prefix_dict.items():
    score_cols=["mhcprime_abt"]
    score_cols+=build_score_cols(prefixes, seeds)
    print(d, len(score_cols), score_cols)
    dataset_score_cols[d]=score_cols
    rank_score_cols=[f"{sc}_rank_r_chp" for sc in score_cols]
    # print(len(rank_score_cols), rank_score_cols)
    dataset_rank_score_cols[d]=rank_score_cols

yd 19 ['mhcprime_abt', 'holec_ft_steps_300_lr_2eN4_seed_0', 'holec_ft_steps_300_lr_2eN4_seed_1', 'holec_ft_steps_300_lr_2eN4_seed_2', 'holec_ft_steps_300_lr_2eN4_seed_3', 'holec_ft_steps_300_lr_2eN4_seed_4', 'holec_ft_steps_300_lr_2eN4_seed_5', 'holec_ft_steps_300_lr_2eN4_seed_6', 'holec_ft_steps_300_lr_2eN4_seed_7', 'holec_ft_steps_300_lr_2eN4_seed_8', 'holec_pepcl_r_dr_steps_300_lr_2eN4_seed_0', 'holec_pepcl_r_dr_steps_300_lr_2eN4_seed_1', 'holec_pepcl_r_dr_steps_300_lr_2eN4_seed_2', 'holec_pepcl_r_dr_steps_300_lr_2eN4_seed_3', 'holec_pepcl_r_dr_steps_300_lr_2eN4_seed_4', 'holec_pepcl_r_dr_steps_300_lr_2eN4_seed_5', 'holec_pepcl_r_dr_steps_300_lr_2eN4_seed_6', 'holec_pepcl_r_dr_steps_300_lr_2eN4_seed_7', 'holec_pepcl_r_dr_steps_300_lr_2eN4_seed_8']
episcan_r9 19 ['mhcprime_abt', 'episcan_r9_ft_steps_300_lr_2eN4_seed_0', 'episcan_r9_ft_steps_300_lr_2eN4_seed_1', 'episcan_r9_ft_steps_300_lr_2eN4_seed_2', 'episcan_r9_ft_steps_300_lr_2eN4_seed_3', 'episcan_r9_ft_steps_300_lr_2eN4_seed_4'

#### Split out method groups and clean each allele at the peptide level

In [ ]:
# seq len helper
def add_seq_len(df):
    df["seq_len"]=df.seq.str.replace("$", "").apply(len)
    return df

In [ ]:
# get per dataset train/val/test sets
train_val_test_per_dataset={
    "yd": {"train": holec_train_set, "val": holec_val_set, "test": holec_test_set},
    "episcan_r9": {"train": episcan_train, "val": episcan_val, "test": episcan_test},
    "episcan_r9rt": {"train": episcan_train, "val": episcan_val, "test": episcan_test}, # duplicate for simplicity
    "ba_con": {"train": ba_train, "val": ba_val, "test": ba_test},
    "escape_con": {"train": escape_train, "val": escape_val, "test": escape_test},
}

# get per dataset iedb data (dup episcan)
iedb_t_cell_data_f_cleaned_episcan_r9=iedb_t_cell_data_f_cleaned_episcan.copy()
iedb_t_cell_data_f_cleaned_episcan_r9rt=iedb_t_cell_data_f_cleaned_episcan.copy()

dataset_iedb_dfs={
    "yd": iedb_t_cell_data_f_cleaned_holec,
    "episcan_r9": iedb_t_cell_data_f_cleaned_episcan_r9,
    "episcan_r9rt": iedb_t_cell_data_f_cleaned_episcan_r9rt,
    "ba_con": iedb_t_cell_data_f_cleaned_ba,
    "escape_con": iedb_t_cell_data_f_cleaned_escape,
}

# datasets and method groups
datasets=["yd", "episcan_r9", "episcan_r9rt", "ba_con", "escape_con"]
recognition_methods=["multimer/tetramer", "binding assay", "SPR"] # prioritization of peptide-mhc-tcr recognized complexes
activation_methods=["ELISPOT", "ICS", "51Cr", "CFSE"] # for functional t cell responses

dataset_cleaned_and_grouped_iedb_data={}

for dataset in datasets:
    df=dataset_iedb_dfs[dataset].copy()
    df=add_seq_len(df)
    
    dataset_train=train_val_test_per_dataset[dataset]["train"]
    dataset_val=train_val_test_per_dataset[dataset]["val"]
    dataset_test=train_val_test_per_dataset[dataset]["test"]

    dataset_train=add_seq_len(dataset_train)
    dataset_val=add_seq_len(dataset_val)
    dataset_test=add_seq_len(dataset_test)

    # filter to matched and unmatched allele x length based on dataset
    if dataset in ["yd", "episcan_r9", "episcan_r9rt", "escape_con"]:
        df_matched_allele_length=df[df.allele.isin(dataset_train.allele.unique())].query("seq_len == 9")
    else:
        # for ba_con
        df_matched_allele_length, _=filter_by_allele_len_pairs(df, dataset_train, allele_col="allele", seq_col="seq", filter_both=False)

    # get pos only
    df_matched_allele_length=df_matched_allele_length.query("label == 1")

    # split activation and recognition methods
    df_matched_allele_length_recog=df_matched_allele_length[df_matched_allele_length.method.isin(recognition_methods)].drop_duplicates(subset=["seq_allele"]).copy()
    df_matched_allele_length_activation=df_matched_allele_length[df_matched_allele_length.method.isin(activation_methods)].drop_duplicates(subset=["seq_allele"]).copy()

    # recognition outputs
    seq_overlap=df_col_set_intersection(dataset_train, df_matched_allele_length_recog, "seq", print_output=True, return_output=True)
    df_matched_allele_length_recog=df_matched_allele_length_recog[~df_matched_allele_length_recog.seq.isin(seq_overlap)]
    seq_overlap=df_col_set_intersection(dataset_val, df_matched_allele_length_recog, "seq", print_output=True, return_output=True)
    df_matched_allele_length_recog=df_matched_allele_length_recog[~df_matched_allele_length_recog.seq.isin(seq_overlap)]
    seq_overlap=df_col_set_intersection(dataset_test, df_matched_allele_length_recog, "seq", print_output=True, return_output=True)
    df_matched_allele_length_recog=df_matched_allele_length_recog[~df_matched_allele_length_recog.seq.isin(seq_overlap)]

    # activation outputs
    seq_overlap=df_col_set_intersection(dataset_train, df_matched_allele_length_activation, "seq", print_output=True, return_output=True)
    df_matched_allele_length_activation=df_matched_allele_length_activation[~df_matched_allele_length_activation.seq.isin(seq_overlap)]
    seq_overlap=df_col_set_intersection(dataset_val, df_matched_allele_length_activation, "seq", print_output=True, return_output=True)
    df_matched_allele_length_activation=df_matched_allele_length_activation[~df_matched_allele_length_activation.seq.isin(seq_overlap)]
    seq_overlap=df_col_set_intersection(dataset_test, df_matched_allele_length_activation, "seq", print_output=True, return_output=True)
    df_matched_allele_length_activation=df_matched_allele_length_activation[~df_matched_allele_length_activation.seq.isin(seq_overlap)]

    print(dataset, "recognition and activation dfs")
    print_data_stats(df_matched_allele_length_recog)
    print_data_stats(df_matched_allele_length_activation)
    print(end="\n")

    dataset_cleaned_and_grouped_iedb_data[dataset]={"recognition": df_matched_allele_length_recog, "activation": df_matched_allele_length_activation}

0
0
0
0
0
0
yd recognition and activation dfs
Total: 2,725
Pos: 2,725 | 100.00%
Neg: 0 | 0.00%
N alleles: 16
Total: 2,927
Pos: 2,927 | 100.00%
Neg: 0 | 0.00%
N alleles: 15

18
2
1
18
0
0
episcan_r9 recognition and activation dfs
Total: 1,584
Pos: 1,584 | 100.00%
Neg: 0 | 0.00%
N alleles: 4
Total: 1,207
Pos: 1,207 | 100.00%
Neg: 0 | 0.00%
N alleles: 4

18
2
1
18
0
0
episcan_r9rt recognition and activation dfs
Total: 1,584
Pos: 1,584 | 100.00%
Neg: 0 | 0.00%
N alleles: 4
Total: 1,207
Pos: 1,207 | 100.00%
Neg: 0 | 0.00%
N alleles: 4

106
0
3
148
0
2
ba_con recognition and activation dfs
Total: 4,481
Pos: 4,481 | 100.00%
Neg: 0 | 0.00%
N alleles: 55
Total: 4,759
Pos: 4,759 | 100.00%
Neg: 0 | 0.00%
N alleles: 77

0
0
0
0
0
0
escape_con recognition and activation dfs
Total: 3,828
Pos: 3,828 | 100.00%
Neg: 0 | 0.00%
N alleles: 35
Total: 4,408
Pos: 4,408 | 100.00%
Neg: 0 | 0.00%
N alleles: 41



#### Compute recovery

In [ ]:
# get recovery of peptides
datasets=["yd", "episcan_r9", "episcan_r9rt", "ba_con", "escape_con"]

# keep separate, easier for plotting
# imm_recovery_over_thresholds_grouped={}
imm_recovery_over_thresholds_recog={}
imm_recovery_over_thresholds_activ={}

for dataset in datasets:

    df_matched_allele_length_recog=dataset_cleaned_and_grouped_iedb_data[dataset]["recognition"].copy()
    df_matched_allele_length_activation=dataset_cleaned_and_grouped_iedb_data[dataset]["activation"].copy()

    n=0
    df_matched_allele_length_recog=filter_alleles_min_samples(df_matched_allele_length_recog, n, allele_col="allele")
    df_matched_allele_length_activation=filter_alleles_min_samples(df_matched_allele_length_activation, n, allele_col="allele")


    print("Recog. alleles", df_matched_allele_length_recog.allele.nunique())
    print("Activ. alleles", df_matched_allele_length_activation.allele.nunique())

    dataset_prefix_map={
        "mhcprime": "mhcprime_abt",
        "ft": dataset_prefix_dict[dataset][0],
        "pepcl": dataset_prefix_dict[dataset][1],
    }

    print(dataset_prefix_map)

    thresholds=[90, 95, 98, 99, 99.5]
    base_col=None
    per_allele=False

    recog_recovery_df=recovery_table_over_thresholds(df_matched_allele_length_recog, dataset_rank_score_cols[dataset], thresholds, dataset_prefix_map, allele_col="allele", base_col=base_col, per_allele=per_allele)

    activ_recovery_df=recovery_table_over_thresholds(df_matched_allele_length_activation, dataset_rank_score_cols[dataset], thresholds, dataset_prefix_map, allele_col="allele", base_col=base_col, per_allele=per_allele)

    imm_recovery_over_thresholds_recog[dataset]=recog_recovery_df
    imm_recovery_over_thresholds_activ[dataset]=activ_recovery_df

Recog. alleles 16
Activ. alleles 15
{'mhcprime': 'mhcprime_abt', 'ft': 'holec_ft_steps_300_lr_2eN4', 'pepcl': 'holec_pepcl_r_dr_steps_300_lr_2eN4'}
Recog. alleles 4
Activ. alleles 4
{'mhcprime': 'mhcprime_abt', 'ft': 'episcan_r9_ft_steps_300_lr_2eN4', 'pepcl': 'episcan_r9_pepcl_r_dr_steps_300_lr_2eN4'}
Recog. alleles 4
Activ. alleles 4
{'mhcprime': 'mhcprime_abt', 'ft': 'episcan_r9rt_ft_steps_400_lr_2eN4', 'pepcl': 'episcan_r9rt_pepcl_r_dr_steps_300_lr_2eN4'}
Recog. alleles 55
Activ. alleles 77
{'mhcprime': 'mhcprime_abt', 'ft': 'ba_con_ft_steps_200_lr_2eN4', 'pepcl': 'ba_con_pepcl_r_dr_steps_300_lr_2eN4'}
Recog. alleles 35
Activ. alleles 41
{'mhcprime': 'mhcprime_abt', 'ft': 'escape_con_ft_steps_200_lr_2eN4', 'pepcl': 'escape_con_pepcl_r_dr_steps_300_lr_2eN4'}


In [ ]:
# save immunogenicity results
t_cell_recovery_results={
    "imm_recovery_over_thresholds_recog": imm_recovery_over_thresholds_recog,
    "imm_recovery_over_thresholds_activ": imm_recovery_over_thresholds_activ,

}
# save_dict_pickle(t_cell_recovery_results, f"{DATA_DIR}/Revisions_Data/t_cell_recovery_results.pkl")